# 🚀 ACR-AGI-3 Official Kaggle Submission Notebook

本ノートブックは ARC Prize 2026 - ARC-AGI-3 コンペティションの公式提出ノートブックです。

- **コンペ仕様**: ARC Gateway インタラクティブゲームプレイ (Simulation Competition)
- **提出仕様**: `/kaggle/working/submission.parquet` (`columns=['row_id', 'game_id', 'end_of_game', 'score']`)
- **実行モード**: 通常コミット時はダミー生成、提出（Rerun）時は Gateway と連携して全タスクを自律プレイ

In [ ]:
# === ARC-AGI-3 公式環境セットアップ（オフライン対応） ===
import os
import subprocess
from pathlib import Path

wheel_dir = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheel_dir.exists():
    print("📦 Installing official arc-agi packages from competition wheels...")
    cmd = [
        "pip", "install", "--no-index", "--find-links", str(wheel_dir),
        "arc-agi", "python-dotenv"
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print("✅ Successfully installed arc-agi and dependencies!")
    else:
        print(f"⚠️ pip notice: {res.stderr[:200]}")
else:
    print("ℹ️ Wheels directory not found (running in local / dataset-only mode).")


In [ ]:
%%writefile /kaggle/working/my_agent.py
import collections
from collections import deque, Counter
import dataclasses
import math
import os
import random
import sys
import time
from typing import Any, Dict, List, Optional, Set, Tuple
from pathlib import Path

try:
    from arcengine import FrameData, GameAction, GameState
except ImportError:
    from enum import Enum
    FrameData = Any
    class GameState(str, Enum):
        NOT_PLAYED = "NOT_PLAYED"
        NOT_FINISHED = "NOT_FINISHED"
        WIN = "WIN"
        GAME_OVER = "GAME_OVER"

    class GameAction(Enum):
        RESET = 0
        ACTION1 = 1
        ACTION2 = 2
        ACTION3 = 3
        ACTION4 = 4
        ACTION5 = 5
        ACTION6 = 6
        ACTION7 = 7

        def is_simple(self):
            return self.value != 6
        def is_complex(self):
            return self.value == 6
        def set_data(self, d):
            self.action_data = d
        @classmethod
        def from_id(cls, i):
            for a in cls:
                if a.value == i:
                    return a
            return cls.ACTION1

try:
    from agents.agent import Agent
except ImportError:
    Agent = object


@dataclasses.dataclass
class VisualObject:
    color: int
    pixels: List[Tuple[int, int]]
    bbox: Tuple[int, int, int, int]
    center_x: int
    center_y: int
    role: str = "unknown"


class GestaltPerceiver:
    def __init__(self) -> None:
        self.prev_grid: Optional[List[List[int]]] = None
        self.player_pos: Optional[Tuple[int, int]] = None
        self.background_color: int = 0

    def parse(self, grid: Any) -> Dict[str, Any]:
        if not grid:
            return {"background_color": 0, "objects": [], "player": None, "targets": [], "walls": set()}

        # 3次元 (N, H, W) やアニメーションシーケンスの安全な正規化（最新フレームを採用）
        if isinstance(grid, (list, tuple)) and len(grid) > 0:
            if isinstance(grid[0], (list, tuple)) and len(grid[0]) > 0 and isinstance(grid[0][0], (list, tuple)):
                grid = grid[-1]
            elif len(grid) == 1 and isinstance(grid[0], (list, tuple)):
                grid = grid[0]

        h = len(grid)
        w = len(grid[0]) if h > 0 and isinstance(grid[0], (list, tuple)) else 0
        if h == 0 or w == 0:
            return {"background_color": 0, "objects": [], "player": None, "targets": [], "walls": set()}

        norm_grid: List[List[int]] = []
        for r in range(h):
            row = []
            for c in range(w):
                val = grid[r][c]
                pixel_val = val[0] if isinstance(val, (list, tuple)) else val
                try:
                    row.append(int(pixel_val))
                except Exception:
                    row.append(0)
            norm_grid.append(row)
        grid = norm_grid

        color_counts: Dict[int, int] = collections.defaultdict(int)
        for r in range(h):
            for c in range(w):
                color_counts[grid[r][c]] += 1
        self.background_color = max(color_counts, key=color_counts.get)

        diff_pixels: List[Tuple[int, int]] = []
        if self.prev_grid and len(self.prev_grid) == h and len(self.prev_grid[0]) == w:
            for r in range(h):
                for c in range(w):
                    if grid[r][c] != self.prev_grid[r][c]:
                        diff_pixels.append((r, c))

        # 3. 連結成分（4-Connected Components）によるスプライト/オブジェクト同定
        visited: Set[Tuple[int, int]] = set()
        raw_components: List[Tuple[int, List[Tuple[int, int]]]] = []

        for r in range(h):
            for c in range(w):
                val = grid[r][c]
                if val == self.background_color or (r, c) in visited:
                    continue

                comp: List[Tuple[int, int]] = []
                q = collections.deque([(r, c)])
                visited.add((r, c))
                while q:
                    cr, cc = q.popleft()
                    comp.append((cr, cc))
                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w:
                            if (nr, nc) not in visited and grid[nr][nc] == val:
                                visited.add((nr, nc))
                                q.append((nr, nc))
                raw_components.append((val, comp))

        objects: List[VisualObject] = []
        walls: Set[Tuple[int, int]] = set()
        targets: List[VisualObject] = []
        identified_player: Optional[Tuple[int, int]] = None

        if diff_pixels:
            non_bg_diff = [p for p in diff_pixels if grid[p[0]][p[1]] != self.background_color]
            if non_bg_diff:
                identified_player = non_bg_diff[0]

        for color, pixs in raw_components:
            if len(pixs) > (h * w * 0.25):
                for p in pixs:
                    walls.add(p)
                continue

            min_r = min(p[0] for p in pixs)
            max_r = max(p[0] for p in pixs)
            min_c = min(p[1] for p in pixs)
            max_c = max(p[1] for p in pixs)
            center_r = (min_r + max_r) // 2
            center_c = (min_c + max_c) // 2

            obj = VisualObject(
                color=color,
                pixels=pixs,
                bbox=(min_r, min_c, max_r, max_c),
                center_x=center_c,
                center_y=center_r,
            )

            if identified_player and (min_r <= identified_player[0] <= max_r) and (min_c <= identified_player[1] <= max_c):
                obj.role = "player"
                self.player_pos = (center_r, center_c)
            else:
                obj.role = "target"
                targets.append(obj)

            objects.append(obj)

        if not self.player_pos and objects:
            self.player_pos = (objects[0].center_y, objects[0].center_x)

        self.prev_grid = [row[:] for row in grid]
        return {
            "background_color": self.background_color,
            "objects": objects,
            "player": self.player_pos,
            "targets": targets,
            "walls": walls,
        }


class GestaltVCGTPlanner:
    def __init__(self, game_id: str = "") -> None:
        self.game_id = game_id
        self.perceiver = GestaltPerceiver()
        self.plan_queue: collections.deque[int] = collections.deque()
        self.clicked_coords: Set[Tuple[int, int]] = set()
        self.known_obstacles: Set[Tuple[int, int]] = set()
        self.last_action_id: Optional[int] = None
        self.step_index: int = 0
        self.confirmed_interactive_group: Optional[Tuple[int, int]] = None
        self.last_clicked_group_key: Optional[Tuple[int, int]] = None
        self.last_clicked_pos: Optional[Tuple[int, int]] = None
        self.last_hit_pos: Optional[Tuple[int, int]] = None

    def decide_action(
        self,
        grid: List[List[int]],
        available_action_ids: List[int],
    ) -> Tuple[int, Dict[str, Any], str]:
        self.step_index += 1
        # 3次元テンソルを最新フレームの 2D グリッドへ完全正規化
        if isinstance(grid, (list, tuple)) and len(grid) > 0:
            if isinstance(grid[0], (list, tuple)) and len(grid[0]) > 0 and isinstance(grid[0][0], (list, tuple)):
                grid = grid[-1]
            elif len(grid) == 1 and isinstance(grid[0], (list, tuple)):
                grid = grid[0]

        parsed = self.perceiver.parse(grid)
        player = parsed.get("player")
        targets: List[VisualObject] = parsed.get("targets", [])
        walls: Set[Tuple[int, int]] = parsed.get("walls", set())
        all_obstacles = walls | self.known_obstacles

        h = len(grid) if grid else 64
        w = len(grid[0]) if grid and grid[0] else 64

        # 1. クリック系タスク (ACTION6)
        if 6 in available_action_ids:
            target_candidates: List[Tuple[Tuple[int, int], Optional[Tuple[int, int]]]] = []

            # ゲシュタルト同定 (A): 同一色・同一サイズの反復スプライト群（キーパッド/タイル盤）を最優先
            size_color_groups = collections.defaultdict(list)
            for obj in targets:
                if 4 <= len(obj.pixels) < (h * w * 0.2):
                    size_color_groups[(obj.color, len(obj.pixels))].append(obj)

            # 過去にヒットが確認された正解グループを絶対最優先！
            if self.confirmed_interactive_group and self.confirmed_interactive_group in size_color_groups:
                hit_group = size_color_groups[self.confirmed_interactive_group]
                if self.last_hit_pos:
                    lx, ly = self.last_hit_pos
                    hit_group = sorted(
                        hit_group,
                        key=lambda o: abs(o.bbox[1] - lx) + abs(o.bbox[0] - ly),
                    )
                for obj in hit_group:
                    for pt in [(obj.bbox[1], obj.bbox[0]), (obj.center_x, obj.center_y)]:
                        if pt not in self.clicked_coords and not any(pt == c[0] for c in target_candidates):
                            target_candidates.append((pt, self.confirmed_interactive_group))

            if not target_candidates:
                repeated_groups = [(k, g) for k, g in size_color_groups.items() if len(g) >= 3]
                repeated_groups.sort(key=lambda item: (-len(item[1]), -len(item[1][0].pixels)))

                for grp_key, group in repeated_groups:
                    for obj in group:
                        for pt in [(obj.bbox[1], obj.bbox[0]), (obj.center_x, obj.center_y)]:
                            if pt not in self.clicked_coords and not any(pt == c[0] for c in target_candidates):
                                target_candidates.append((pt, grp_key))

            # ゲシュタルト同定 (B): その他のターゲットオブジェクト
            if not target_candidates:
                sorted_targets = sorted(targets, key=lambda o: len(o.pixels))
                for obj in sorted_targets:
                    grp_key = (obj.color, len(obj.pixels))
                    for pt in [(obj.bbox[1], obj.bbox[0]), (obj.center_x, obj.center_y)]:
                        if pt not in self.clicked_coords and not any(pt == c[0] for c in target_candidates):
                            target_candidates.append((pt, grp_key))

            # ゲシュタルト同定 (C): 各オブジェクトの構成ピクセル
            if not target_candidates:
                for obj in targets:
                    grp_key = (obj.color, len(obj.pixels))
                    for r, c in obj.pixels:
                        if (c, r) not in self.clicked_coords and not any((c, r) == cand[0] for cand in target_candidates):
                            target_candidates.append(((c, r), grp_key))
                            break

            if target_candidates:
                (target_x, target_y), grp_key = target_candidates[0]
                self.clicked_coords.add((target_x, target_y))
                self.last_action_id = 6
                self.last_clicked_group_key = grp_key
                self.last_clicked_pos = (target_x, target_y)
                return 6, {"x": int(target_x), "y": int(target_y)}, f"VCGT Click: Target at ({target_x}, {target_y})"

            bg = parsed.get("background_color", 0)
            for r in range(h):
                for c in range(w):
                    if grid[r][c] != bg and (c, r) not in self.clicked_coords:
                        self.clicked_coords.add((c, r))
                        self.last_action_id = 6
                        self.last_clicked_group_key = None
                        return 6, {"x": int(c), "y": int(r)}, f"VCGT Click: Pixel at ({c}, {r})"

            cx, cy = w // 2, h // 2
            return 6, {"x": int(cx), "y": int(cy)}, f"VCGT Click: Center ({cx}, {cy})"

        # 2. 移動系タスク (ACTION1〜5): サブゴール経路計画
        if self.plan_queue:
            act = self.plan_queue.popleft()
            if act in available_action_ids:
                self.last_action_id = act
                return act, {}, f"VCGT Macro: Step {self.step_index} pursuing path"

        if player and targets:
            nearest_target = min(
                targets,
                key=lambda obj: abs(obj.center_y - player[0]) + abs(obj.center_x - player[1]),
            )
            goal_r, goal_c = nearest_target.center_y, nearest_target.center_x

            actions_plan = self._find_path_bfs(
                start=player,
                goal=(goal_r, goal_c),
                obstacles=all_obstacles,
                grid_shape=(h, w),
                available_actions=available_action_ids,
            )

            if actions_plan:
                for a in actions_plan:
                    self.plan_queue.append(a)
                act = self.plan_queue.popleft()
                self.last_action_id = act
                return act, {}, f"VCGT Plan: Heading to ({goal_c}, {goal_r})"

        move_actions = [a for a in available_action_ids if a in [1, 2, 3, 4]]
        if move_actions:
            act = self.last_action_id if self.last_action_id in move_actions else move_actions[0]
            self.last_action_id = act
            return act, {}, f"VCGT Momentum: {act}"

        fallback_act = available_action_ids[0] if available_action_ids else 1
        return fallback_act, {}, "VCGT Fallback"

    def on_feedback(self, is_effective: bool, pixels_changed: int) -> None:
        if is_effective and self.last_action_id == 6 and self.last_clicked_group_key:
            self.confirmed_interactive_group = self.last_clicked_group_key
            if self.last_clicked_pos:
                self.last_hit_pos = self.last_clicked_pos
        if not is_effective and self.last_action_id in [1, 2, 3, 4]:
            self.plan_queue.clear()
            self.last_action_id = None

    def _find_path_bfs(
        self,
        start: Tuple[int, int],
        goal: Tuple[int, int],
        obstacles: Set[Tuple[int, int]],
        grid_shape: Tuple[int, int],
        available_actions: List[int],
    ) -> List[int]:
        h, w = grid_shape
        moves: List[Tuple[int, int, int]] = []
        if 1 in available_actions:
            moves.append((1, -1, 0))  # UP
        if 2 in available_actions:
            moves.append((2, 1, 0))   # DOWN
        if 3 in available_actions:
            moves.append((3, 0, -1))  # LEFT
        if 4 in available_actions:
            moves.append((4, 0, 1))   # RIGHT

        if not moves:
            return []

        queue = collections.deque([(start, [])])
        visited = {start}

        while queue:
            (curr_r, curr_c), path = queue.popleft()
            if (curr_r, curr_c) == goal:
                return path

            for act_id, dr, dc in moves:
                nr, nc = curr_r + dr, curr_c + dc
                if 0 <= nr < h and 0 <= nc < w and (nr, nc) not in visited and (nr, nc) not in obstacles:
                    visited.add((nr, nc))
                    queue.append(((nr, nc), path + [act_id]))
                    if len(path) >= 25:
                        return path + [act_id]
        return []


class MyAgent(Agent):
    """ACR-AGI-3 VCGT 準拠 思考型メタエージェント (Gestalt-VCGT Agent)."""

    MAX_ACTIONS = 80

    def __init__(
        self,
        card_id: str = "local",
        game_id: str = "default",
        agent_name: str = "MyAgent",
        ROOT_URL: str = "http://local",
        record: bool = False,
        arc_env: Any = None,
        *args: Any,
        **kwargs: Any,
    ) -> None:
        try:
            super().__init__(card_id, game_id, agent_name, ROOT_URL, record, arc_env, *args, **kwargs)
        except Exception:
            pass
        self.game_id = game_id or getattr(self, "game_id", "default")
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        self.step_count = 0
        self.action_history: List[int] = []
        self.last_frame_hash: Optional[int] = None
        self.stuck_count: int = 0
        self.planner = GestaltVCGTPlanner(game_id=self.game_id)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        state = getattr(latest_frame, "state", None)
        return state is GameState.WIN

    def _get_cands(self, latest_frame: FrameData) -> List[Any]:
        avail = getattr(latest_frame, "available_actions", None)
        reset_val = getattr(GameAction.RESET, "value", 0)
        cands = []
        if avail:
            for act_id in avail:
                if act_id != reset_val:
                    try:
                        cands.append(GameAction.from_id(act_id))
                    except Exception:
                        pass
        if not cands:
            all_actions = list(GameAction) if hasattr(GameAction, "__iter__") else [
                getattr(GameAction, f"ACTION{i}", None) for i in range(1, 8)
            ]
            cands = [a for a in all_actions if a is not None and getattr(a, "value", -1) != reset_val]
        return cands

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> Any:
        self.step_count += 1
        state = getattr(latest_frame, "state", None)

        if state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.step_count = 0
            self.stuck_count = 0
            self.action_history.clear()
            self.planner = GestaltVCGTPlanner(game_id=self.game_id)
            return GameAction.RESET

        try:
            cands = self._get_cands(latest_frame)
            if not cands:
                return GameAction.RESET

            grid = getattr(latest_frame, "frame", [])
            cand_ids = [getattr(a, "value", 1) for a in cands]

            # 前ステップの差分フィードバック
            def _hash_grid(g):
                try:
                    if not g:
                        return 0
                    if isinstance(g, (list, tuple)) and len(g) > 0:
                        if isinstance(g[0], (list, tuple)) and len(g[0]) > 0 and isinstance(g[0][0], (list, tuple)):
                            g = g[-1]
                        elif len(g) == 1 and isinstance(g[0], (list, tuple)):
                            g = g[0]
                    return hash(tuple(tuple(int(c[0]) if isinstance(c, (list, tuple)) else int(c) for c in row) for row in g))
                except Exception:
                    return 0

            current_hash = _hash_grid(grid)
            is_eff = (self.last_frame_hash is not None and current_hash != self.last_frame_hash)
            self.planner.on_feedback(is_effective=is_eff, pixels_changed=1 if is_eff else 0)
            self.last_frame_hash = current_hash

            # VCGT メタスキルによる人間的計画思考
            act_id, act_data, reasoning = self.planner.decide_action(grid, cand_ids)
            chosen_action = GameAction.from_id(act_id)

            if hasattr(chosen_action, "is_complex") and chosen_action.is_complex():
                chosen_action.set_data(act_data)
                chosen_action.reasoning = {
                    "desired_action": f"{chosen_action.value}",
                    "my_reason": reasoning,
                }
            else:
                chosen_action.reasoning = reasoning

            self.action_history.append(act_id)
            return chosen_action

        except Exception as e:
            print(f"[DEBUG MyAgent Error] {type(e).__name__}: {e}")
            avail = getattr(latest_frame, "available_actions", None)
            if avail:
                act_id = [x for x in avail if x != 0][0] if any(x != 0 for x in avail) else 0
                return GameAction.from_id(act_id)
            return GameAction.from_id(1)


In [ ]:
# === Rerun モード: ARC Gateway 連携ゲームプレイ ===
import os
import subprocess
from pathlib import Path

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("🌐 [RERUN MODE] Waiting for ARC Gateway to be ready...")
    # 1. Gateway の起動待機
    subprocess.run([
        "curl", "--fail", "--retry", "999", "--retry-all-errors", "--retry-delay", "5",
        "--retry-max-time", "600", "http://gateway:8001/api/games"
    ], check=True)
    print("✅ Gateway is live and responding!")

    # 2. ARC-AGI-3-Agents のセットアップ
    agents_src = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    agents_dest = Path("/kaggle/working/ARC-AGI-3-Agents")
    if not agents_dest.exists() and agents_src.exists():
        import shutil
        shutil.copytree(agents_src, agents_dest)
        print("✅ Copied ARC-AGI-3-Agents to /kaggle/working")

    # 3. エージェントの配置
    my_agent_src = Path("/kaggle/working/my_agent.py")
    if my_agent_src.exists() and agents_dest.exists():
        import shutil
        shutil.copy(my_agent_src, agents_dest / "agents" / "templates" / "my_agent.py")

    # 4. 最小構成の __init__.py (余分な langgraph 依存を回避)
    init_content = """from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
"""
    if agents_dest.exists():
        with open(agents_dest / "agents" / "__init__.py", "w", encoding="utf-8") as f:
            f.write(init_content)

    # 5. .env のオーバーライド設定
    env_content = """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
"""
    if agents_dest.exists():
        with open(agents_dest / ".env", "w", encoding="utf-8") as f:
            f.write(env_content)

    # 6. エージェント実行
    print("🚀 Running agent against Gateway...")
    env = os.environ.copy()
    env["MPLBACKEND"] = "agg"
    res = subprocess.run(
        ["python", "main.py", "--agent", "myagent"],
        cwd=str(agents_dest),
        env=env,
        capture_output=True,
        text=True
    )
    if res.stdout:
        print("Agent STDOUT (tail):")
        print(res.stdout[-2000:])
    if res.stderr:
        print("Agent STDERR (tail):")
        print(res.stderr[-1000:])
    print("✅ Gateway game session completed successfully!")
else:
    print("🧪 [STANDALONE / COMMIT MODE] Skipping gateway run.")


In [ ]:
# === 提出ファイル生成 (submission.parquet / submission.csv) ===
import os
import pandas as pd
from pathlib import Path

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
parquet_path = working_dir / "submission.parquet"
csv_path = working_dir / "submission.csv"

# 非 Rerun モード（コミット時）または Rerun 完了時の安全策として生成
if not parquet_path.exists() or not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    )
    submission.to_parquet(parquet_path, index=False)
    submission.to_csv(csv_path, index=False)
    print(f"✅ Generated submission for Kaggle leaderboard: {parquet_path}")

print(f"Submission status: exists={parquet_path.exists()}, size={parquet_path.stat().st_size if parquet_path.exists() else 0} bytes")


In [ ]:
# === 提出ファイルのバリデーション検証 ===
import pandas as pd
from pathlib import Path

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
parquet_path = working_dir / "submission.parquet"

assert parquet_path.exists(), f"❌ {parquet_path} was not created!"
df = pd.read_parquet(parquet_path)

print("=== 📊 Submission Artifacts Verification ===")
print(f"Parquet File: {parquet_path} ({parquet_path.stat().st_size} bytes)")
print(f"Columns: {list(df.columns)}")
print(f"Rows: {len(df)}")
print(df.head())

assert list(df.columns) == ["row_id", "game_id", "end_of_game", "score"], f"Invalid columns: {list(df.columns)}"
assert len(df) > 0, "Submission dataframe is empty!"
print("\n🎉 Official ARC-AGI-3 submission verified successfully! Ready for Leaderboard!")
